In [6]:
import pandas as pd

df_positive = pd.read_parquet("/Users/yavuzlule/Desktop/bsc-relish/notebooks/relish_chunked.parquet")
df_negative = pd.read_parquet("/Users/yavuzlule/Desktop/bsc-relish/notebooks/wikipedia_12000_chunks-test.parquet")

In [10]:
df_positive.head()
df_positive["label"] = 1

In [11]:
df_negative.head()

,original_row,chunk_index,chunk_text,LANGUAGE,label
0,300,0,Las comelináceas (nombre científico Commelinac...,Spanish,0
1,300,1,(con forma de V en el corte transversal). Much...,Spanish,0
2,300,2,"estigma, capitado, con flecos, o 3-lobado. 3 l...",Spanish,0
3,300,3,los géneros de Commelinaceae pertenecen a dos ...,Spanish,0
4,300,4,Hassk. Buforrestia C.B.Clarke Callisia Loefl. ...,Spanish,0


In [ ]:
df_positive.rename(columns={"chunk_id": "chunk_index"}, inplace=True)


In [15]:
df_positive.head()

,title,text,translation,LANGUAGE,chunk_text,chunk_index,label
0,How you want to make a food of hens,3 lb chicken 3/4 t anise in eggs 12 threads sa...,Ginger-saffron chicken 3 pounds chicken teaspo...,English,3 lb chicken 3/4 t anise in eggs 12 threads sa...,0.0,1
1,To make cheesecakes,"Take 12 quarts of milk warm from the cow, turn...",Combine 12 quarts of warm milk with a generous...,English,"Take 12 quarts of milk warm from the cow, turn...",0.0,1
2,To make cheesecakes,"Take 12 quarts of milk warm from the cow, turn...",Combine 12 quarts of warm milk with a generous...,English,70 minutes. Let cool 1 hour before serving.,1.0,1
3,A good filling,4 large granny smith apples dough:1/2 cup flou...,Apple bake with honey glaze **ingredients:** *...,English,4 large granny smith apples dough:1/2 cup flou...,0.0,1
4,Chicken covered with walnuts and saffron,4.8 lbs chicken 1/2 t salt topping: 1 c cilant...,Chicken with walnut & herb meatballs **yields:...,English,4.8 lbs chicken 1/2 t salt topping: 1 c cilant...,0.0,1


In [16]:
df_positive = df_positive[["chunk_index", "chunk_text", "LANGUAGE", "label"]]
df_negative = df_negative[["chunk_index", "chunk_text", "LANGUAGE", "label"]]

final_df = pd.concat([df_positive, df_negative], ignore_index=True)




In [19]:
len(df_positive), len(df_negative), len(final_df)

(8200, 12000, 20200)

In [17]:
final_df.head()

,chunk_index,chunk_text,LANGUAGE,label
0,0.0,3 lb chicken 3/4 t anise in eggs 12 threads sa...,English,1
1,0.0,"Take 12 quarts of milk warm from the cow, turn...",English,1
2,1.0,70 minutes. Let cool 1 hour before serving.,English,1
3,0.0,4 large granny smith apples dough:1/2 cup flou...,English,1
4,0.0,4.8 lbs chicken 1/2 t salt topping: 1 c cilant...,English,1


In [18]:
final_df.describe()

,chunk_index,label
count,20112.000000,20200.000000
mean,8.686605,0.405941
std,12.928539,0.491085
min,0.000000,0.000000
25%,0.000000,0.000000
50%,3.000000,0.000000
75%,12.000000,1.000000
max,103.000000,1.000000


In [20]:
final_df.to_parquet("/Users/yavuzlule/Desktop/bsc-relish/notebooks/multilingual_test_dataset.parquet", index=False)

In [21]:
import pandas as pd

def prepare_finetune_df(df):
    df = df.copy()

    # Keep only valid binary labels
    df = df[df["label"].isin([0, 1])]

    # Ensure text is string and not null
    df["chunk_text"] = df["chunk_text"].fillna("").astype(str)

    # Remove empty / whitespace-only texts
    df = df[df["chunk_text"].str.strip().astype(bool)]

    # Optional: remove extremely short noise samples (tune threshold if needed)
    df = df[df["chunk_text"].str.len() > 5]

    # Reset index for training stability
    df = df.reset_index(drop=True)

    return df

In [22]:
final_df = prepare_finetune_df(final_df)

final_df.describe()

,chunk_index,label
count,20035.000000,20123.000000
mean,8.719890,0.403717
std,12.942181,0.490654
min,0.000000,0.000000
25%,0.000000,0.000000
50%,3.000000,0.000000
75%,12.000000,1.000000
max,103.000000,1.000000


In [23]:


df = final_df.copy()

df = df[df["label"].isin([0, 1])]

df["chunk_text"] = df["chunk_text"].fillna("").astype(str)
df = df[df["chunk_text"].str.strip().astype(bool)]

df["word_count"] = df["chunk_text"].str.split().str.len()

In [25]:
import numpy as np
import pandas as pd

bins = [0, 20, 50, 100, 256, 400, 800, np.inf]
labels = ["0-20", "20-50", "50-100", "100-256", "200-400", "400-800", "800+"]

df["word_bin"] = pd.cut(df["word_count"], bins=bins, labels=labels)

df["word_bin"].value_counts().sort_index()

word_bin
0-20         642
20-50       1400
50-100      2469
100-256    15612
200-400        0
400-800        0
800+           0
Name: count, dtype: int64

In [ ]:
df.to_parquet("/Users/yavuzlule/Desktop/bsc-relish/notebooks/multilingual_test_dataset.parquet", index=False)